**Drs List on Big Query**

In [10]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Drs_List"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [
    # Primary Key
    bigquery.SchemaField(
        "Dr_Code",
        "STRING",
        mode="REQUIRED"
    ),

    # Basic Information
    bigquery.SchemaField("Speciality", "STRING"),
    bigquery.SchemaField("Dr_Name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("Email", "STRING"),

    # Financial Information
    bigquery.SchemaField("Visit_Fee", "INTEGER"),

    # Status
    bigquery.SchemaField("Active_Dr", "STRING"),

    # Dates
    bigquery.SchemaField("Joining_Date", "DATE")
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

# Clustering by speciality and dr_code 
table.clustering_fields = [
    "speciality",
    "dr_code"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Drs_List


**ETL & Fill Drs List by Data**

In [12]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Drs_List"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# READ EXCEL FILE
# =========================================

df = pd.read_excel("../Resources/Drs_List.xlsx")

# =========================================
# GET LAST DR CODE FROM BIGQUERY
# =========================================

query = f"""
SELECT MAX(CAST(REGEXP_EXTRACT(Dr_Code, r'\\d+') AS INT64)) AS max_id
FROM `{TABLE_REF}`
"""

result = client.query(query).to_dataframe()

last_id = result.iloc[0]["max_id"]

if pd.isna(last_id):
    last_id = 0

# =========================================
# GENERATE NEW DR CODES
# =========================================

df.insert(
    0,
    "Dr_Code",
    [f"Dr{i:03d}" for i in range(last_id + 1, last_id + len(df) + 1)]
)

# =========================================
# CLEAN DATA TYPES
# =========================================

df["Joining_Date"] = pd.to_datetime(
    df["Joining_Date"]
).dt.date

df["Visit_Fee"] = pd.to_numeric(
    df["Visit_Fee"],
    errors="coerce"
)

# =========================================
# REORDER COLUMNS TO MATCH TABLE
# =========================================

df = df[
    [
        "Dr_Code",
        "Speciality",
        "Dr_Name",
        "Email",
        "Visit_Fee",
        "Active_Dr",
        "Joining_Date"
    ]
]

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND"
)

job = client.load_table_from_dataframe(
    df,
    TABLE_REF,
    job_config=job_config
)

job.result()

print("=================================")
print(f"{len(df)} rows inserted successfully")
print(TABLE_REF)
print("=================================")

20 rows inserted successfully
depihealthnux.Depihealthnux_Main.Drs_List
